# [5.5] Toy Discrete Diffusion Language Models and Local DiffusionGemma Proof - Solutions

This notebook executes the solved implementations and surfaces the same signature result as the exercise notebook. Keep the claim boundary in view: tiny diffusion mechanics are tested directly; released DiffusionGemma is represented by a pinned local NVFP4 generation proof, not by denoising-time interpretability.

In [ ]:
import json
import sys
from pathlib import Path

import torch as t

chapter = "chapter5_modern_architectures"
section = "part5_diffusion_language_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_diffusion_language_models.tests as tests
import part5_diffusion_language_models.solutions as solutions

linear_mask_schedule = solutions.linear_mask_schedule
expected_mask_fraction = solutions.expected_mask_fraction
apply_forward_noising = solutions.apply_forward_noising
masked_denoising_loss = solutions.masked_denoising_loss
token_entropy = solutions.token_entropy
confidence_remask = solutions.confidence_remask
uniform_remask = solutions.uniform_remask
diffusion_sampler = solutions.diffusion_sampler
commitment_times = solutions.commitment_times
edit_distance = solutions.edit_distance
validate_activation_trajectory = solutions.validate_activation_trajectory
TinyConditionalDiffusionLM = solutions.TinyConditionalDiffusionLM

## Schedule and Noising

<details><summary>Expected output</summary>

```text
All tests in `test_linear_mask_schedule_and_expected_fraction` passed!
All tests in `test_forward_noising_extremes_and_seeded_masks` passed!
```

</details>

<details><summary>Help - the forward direction</summary>

The forward process is low-to-high corruption. The sampler will walk these same probabilities backward.

</details>

In [ ]:
tests.test_linear_mask_schedule_and_expected_fraction(
    linear_mask_schedule,
    expected_mask_fraction,
)
tests.test_forward_noising_extremes_and_seeded_masks(
    apply_forward_noising,
    linear_mask_schedule,
)

## Loss, Remasking, and Oracle Sampling

<details><summary>Expected output</summary>

```text
All tests in `test_masked_denoising_loss_uses_only_masked_positions` passed!
All tests in `test_confidence_remask_entropy_and_uniform_control` passed!
All tests in `test_oracle_diffusion_sampler_recovers_target` passed!
```

</details>

<details><summary>Help - what the controls isolate</summary>

The loss test isolates target leakage; the remasking test isolates the confidence heuristic; the oracle sampler test isolates the reverse-process loop before model training enters the picture.

</details>

In [ ]:
tests.test_masked_denoising_loss_uses_only_masked_positions(masked_denoising_loss)
tests.test_confidence_remask_entropy_and_uniform_control(
    confidence_remask,
    token_entropy,
    uniform_remask,
)
tests.test_oracle_diffusion_sampler_recovers_target(
    diffusion_sampler,
    linear_mask_schedule,
)

## Diagnostics and Tiny Denoiser

<details><summary>Expected output</summary>

```text
All tests in `test_commitment_edit_distance_and_activation_trajectory` passed!
All tests in `test_tiny_conditional_diffusion_lm_forward_shape` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Help - bidirectional does not mean leaked</summary>

The model can attend across the whole corrupted sequence because hidden targets have been replaced by mask tokens. That is different from letting a next-token model see the future answer.

</details>

In [ ]:
tests.test_commitment_edit_distance_and_activation_trajectory(
    commitment_times,
    edit_distance,
    validate_activation_trajectory,
)
tests.test_tiny_conditional_diffusion_lm_forward_shape(TinyConditionalDiffusionLM)
tests.test_notebook_contract(solutions.run_smoke_test)

## Signature Result

<details><summary>Interpreting the signature result</summary>

The tiny CUDA path proves the generated copy-pair diffusion task under controls. The NVFP4 path proves one released-checkpoint local generation route through isolated vLLM. Do not read this as released DiffusionGemma activation patching evidence.

</details>

In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test()
summary = {
    "heldout_masked_accuracy": gpu["heldout_masked_accuracy"],
    "sampler_exact_match": gpu["sampler_exact_match"],
    "shuffled_label_accuracy": gpu["shuffled_label_accuracy"],
    "suffix_commitment_mean_step": gpu["suffix_commitment_mean_step"],
    "entropy_by_step": [round(x, 4) for x in gpu["entropy_by_step"]],
    "diffusiongemma_generation_ready": gpu["diffusiongemma_generation_ready"],
    "nvfp4_isolated_vllm_generation_ready": gpu["diffusiongemma_nvfp4_isolated_vllm_generation_ready"],
    "external_vllm_runtime": {
        "torch": gpu["diffusiongemma_external_vllm_torch_version"],
        "cuda": gpu["diffusiongemma_external_vllm_torch_cuda_version"],
        "vllm": gpu["diffusiongemma_external_vllm_vllm_version"],
    },
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}
summary

## Limitations

The released checkpoint proof is intentionally scoped. It proves local NVFP4 generation on this machine through an isolated runtime. It does not prove denoising-time circuit tracing, patching, throughput, broad quality, or BF16 direct loading on 24GB.